# Fashion-MNIST Dataset - CNN

### [Fashion-MNIST Dataset](https://github.com/zalandoresearch/fashion-mnist)

Author: [Kevin Thomas](mailto:ket189@pitt.edu)

License: MIT

## Citation

[1] Han Xiao, Kashif Rasul, Roland Vollgraf, https://github.com/zalandoresearch/fashion-mnist

## Install Libraries

In [ ]:
# conda activate prod
# conda install -c conda-forge pytorch torchvision torchaudio
# conda install numpy pandas matplotlib scikit-learn

## Import Libraries

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision import transforms
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

## Seed

In [ ]:
SEED = 42
SEED

In [ ]:
torch.manual_seed(SEED)

## Parameters

In [ ]:
IN_CHANNELS = 1
IN_CHANNELS

In [ ]:
NUM_CLASSES = 10
NUM_CLASSES

In [ ]:
IMAGE_SIZE = 28
IMAGE_SIZE

In [ ]:
CLASS_NAMES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
CLASS_NAMES

## Hyperparameters

In [ ]:
LEARNING_RATE = 0.001
LEARNING_RATE

In [ ]:
EPOCHS = 5
EPOCHS

In [ ]:
BATCH_SIZE = 128
BATCH_SIZE

## Device

In [ ]:
DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu")
DEVICE

## Load Dataset

Fashion-MNIST is a real dataset of 70,000 grayscale clothing images. torchvision downloads it once into `data/` and normalizes each image.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))])
train_data = datasets.FashionMNIST('data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST('data', train=False, download=True, transform=transform)
len(train_data), len(test_data)

### Visualize a Sample

In [ ]:
image, label = train_data[0]
plt.imshow(image.squeeze(), cmap='gray')
plt.title(CLASS_NAMES[label])
plt.axis('off')
plt.show()

## Create Model

In [ ]:
class CNN(nn.Module):
    """
    A small convolutional neural network for 28x28 grayscale images.
    """

    def __init__(self, num_classes=NUM_CLASSES):
        """
        Initialize the convolutional and dense layers.

        Parameters:
            num_classes (int): Number of output classes.

        Returns:
            None
        """
        super(CNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes))

    def forward(self, x):
        """
        Run the forward pass of the network.

        Parameters:
            x (torch.Tensor): Batch of images.

        Returns:
            torch.Tensor: Output logits.
        """
        x = self.features(x)
        return self.classifier(x)

## Instantiate Model

In [ ]:
torch.manual_seed(SEED)
model = CNN().to(DEVICE)
model

## Create DataLoaders

In [ ]:
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)
len(train_loader), len(test_loader)

## Create Loss Function & Optimizer

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

## Train Model

### Functions

In [ ]:
def train_epoch(model, loader, loss_fn, optimizer):
    """
    Train the model for a single epoch.

    Parameters:
        model (nn.Module): The model to train.
        loader (DataLoader): Training data loader.
        loss_fn (nn.Module): Loss function.
        optimizer (torch.optim.Optimizer): Parameter update rule.

    Returns:
        float: Mean training loss for the epoch.
    """
    model.train()
    total = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(y)
    return total / len(loader.dataset)


def evaluate(model, loader, loss_fn):
    """
    Evaluate the model over a loader.

    Parameters:
        model (nn.Module): The model to evaluate.
        loader (DataLoader): Evaluation data loader.
        loss_fn (nn.Module): Loss function.

    Returns:
        tuple: Mean loss and accuracy.
    """
    model.eval()
    total, correct = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            total += loss_fn(logits, y).item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
    n = len(loader.dataset)
    return total / n, correct / n

### Training Loop

In [ ]:
history = {'loss': [], 'accuracy': []}
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, loss_fn, optimizer)
    test_loss, test_accuracy = evaluate(model, test_loader, loss_fn)
    history['loss'].append(test_loss)
    history['accuracy'].append(test_accuracy)
    print(f"Epoch {epoch + 1} | train loss {train_loss:.4f} | test accuracy {test_accuracy:.4f}")

### Visualize Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['loss'])
axes[0].set_title('Test loss')
axes[0].set_xlabel('Epoch')
axes[1].plot(history['accuracy'])
axes[1].set_title('Test accuracy')
axes[1].set_xlabel('Epoch')
fig.tight_layout()
plt.show()

## Evaluate Model

In [ ]:
_, accuracy = evaluate(model, test_loader, loss_fn)
print(f"Test accuracy: {accuracy:.4f}")

### Confusion Matrix and Classification Report

In [ ]:
all_preds = []
all_labels = []
model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        all_preds.append(model(x).argmax(1).cpu().numpy())
        all_labels.append(y.numpy())
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)
print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

## Save Model

In [ ]:
torch.save(model.state_dict(), 'cnn_fashion_mnist.pt')
print('saved cnn_fashion_mnist.pt')

## Load Model

In [ ]:
loaded_model = CNN().to(DEVICE)
loaded_model.load_state_dict(torch.load('cnn_fashion_mnist.pt', map_location=DEVICE))
loaded_model.eval()
print('loaded cnn_fashion_mnist.pt')

## Inference

### Function

In [ ]:
def predict(model, image):
    """
    Predict the class of one normalized image tensor.

    Parameters:
        model (nn.Module): Trained model.
        image (torch.Tensor): A 1x28x28 image tensor.

    Returns:
        tuple: Predicted class name and confidence.
    """
    model.eval()
    with torch.no_grad():
        logits = model(image.unsqueeze(0).to(DEVICE))
        probs = torch.softmax(logits, dim=1)
        confidence, predicted = probs.max(dim=1)
    return CLASS_NAMES[predicted.item()], confidence.item()

### Run Inference

In [ ]:
image, label = test_data[7]
name, confidence = predict(loaded_model, image)
print(f"True: {CLASS_NAMES[label]} | Predicted: {name} ({confidence:.4f})")
plt.imshow(image.squeeze(), cmap='gray')
plt.title(f"Predicted: {name}")
plt.axis('off')
plt.show()